In [ ]:
# Stanley Uche Godfrey
# ustan.godfrey@gmail.com
# A Movie Chatbot, The goal is to build a chatbot 
# that understands natural language queries 
# and retrieves relevant movie information from an IMDb dataset.


In [ ]:
!pip install openai pandas numpy faiss-cpu sentence-transformers
!pip install openai pandas chromadb


  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-2026.5.1-cp313-cp313-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached uvloop-0.22.1-cp313-cp313-macosx_10_13_universal2.whl.metadata (4.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 12.3 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 15.0 MB/s  0:00:00eta 0:00:01
Using cached jsonschema-4.26.0-py3-none-any.whl (90 kB)
Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 16.8 MB/s  0:00:00 eta 0:

In [42]:


# Importing the CSV module for reading and writing CSV files.
import csv

# Importing pandas for data manipulation and analysis.
import pandas as pd

# Importing numpy for numerical operations and handling arrays efficiently.
import numpy as np

# Importing os to interact with the operating system, such as environment variables and file paths.
import os

# Importing getpass to securely handle user input (e.g., API keys or passwords).
import getpass

import math

# Importing the OpenAI library to interact with OpenAI's API services.
from openai import OpenAI

# Import basic libraries
import os
from dotenv import load_dotenv



In [48]:
# Store your OpenAI API key
dotenv_dir='../chat_bot/prod_small/' #Replace with your path
print("dotenv_dir",dotenv_dir)  # Debugging statement to check the path to the .env file
env_path=os.path.join(dotenv_dir,'.env')
# Load environment variables from .env file
load_dotenv(env_path)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

print(OPENAI_API_KEY[0:70]+'...')


dotenv_dir ../chat_bot/prod_small/
sk-proj-aHFYu6EPI2h05-gBBSmDLH7AHZ-J05j0tKMrkqgb0mMjbmXzoGi2mFBznEpGZl...


In [49]:
# Load the data
imdb_data = pd.read_csv('IMDb_Dataset.csv')


In [41]:
# View & Understand the data
print(imdb_data.columns.tolist())  # Print column names to understand the structure of the dataset


['Title', 'IMDb Rating', 'Year', 'Certificates', 'Genre', 'Director', 'Star Cast', 'MetaScore', 'Poster-src', 'Duration (minutes)']


In [51]:
# Create movie description for each movie from the details provided in the dataset
movie_description = []
movie_description_len = []
for index, row in imdb_data.iterrows():
    description = f"{row['Title']} is a {row['Genre']} movie directed by {row['Director']}. It stars {row['Star Cast']} and was released in {row['Year']}."
    movie_description.append(description)
    movie_description_len.append(len(description))
imdb_data['description'] = movie_description
imdb_data['description_len'] = movie_description_len


In [ ]:
# Now, data is ready!
# Its time to create your vector store
# Perform Text Chunking


In [ ]:
# Create embeddings for the chunks
# See https://python.langchain.com/docs/integrations/text_embedding/ for a list of available embedding models on LangChain

client = OpenAI()

def get_embedding(text):

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding


In [ ]:
# Create a vector store using the created chunks and the embeddings model

# Importing RecursiveCharacterTextSplitter from LangChain for chunking large text into smaller, manageable pieces.
# This helps in optimizing text for processing and retrieval.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Importing OpenAIEmbeddings from LangChain to generate numerical vector representations (embeddings) of text.
# These embeddings capture the semantic meaning of the text for efficient similarity searches.
from langchain_openai import OpenAIEmbeddings

# Importing FAISS (Facebook AI Similarity Search) from LangChain's community package.
# FAISS is used for storing and retrieving embeddings efficiently by finding similar vectors.
from langchain_community.vectorstores import FAISS

# Split the input text using Recursive Character Chunking
# See this for more details https://python.langchain.com/v0.1/docs/modules/data_connection/document_transformers/recursive_text_splitter/

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=250,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)

documents = text_splitter.create_documents(movie_description)

embeddings = OpenAIEmbeddings()

vector = FAISS.from_documents(documents, embeddings)




<class 'method'>


In [56]:
# Importing ChatOpenAI from LangChain to interact with OpenAI's language models, such as GPT, for generating responses.
from langchain_openai import ChatOpenAI

# Importing ChatPromptTemplate to create structured prompts for the chatbot, ensuring consistent interactions with the AI model.
from langchain_core.prompts import ChatPromptTemplate

# Importing OpenAIEmbeddings to convert text data into numerical vector representations for similarity search and retrieval.
from langchain_openai import OpenAIEmbeddings

# Importing ChatPromptTemplate again (duplicate import, should be removed to avoid redundancy).
from langchain_core.prompts import ChatPromptTemplate

# Importing create_stuff_documents_chain to combine and process retrieved documents for meaningful AI-generated responses.
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Importing create_retrieval_chain to build a chain that retrieves relevant documents from a vector store and generates AI responses.
from langchain_classic.chains import create_retrieval_chain

# Importing StrOutputParser from LangChain to parse the output
from langchain_core.output_parsers import StrOutputParser


In [ ]:
# Create the llm model
llm = ChatOpenAI(api_key=os.environ["OPENAI_API_KEY"], model = 'gpt-5.4-nano')

# Importing the output parser to process and format the model's response into a readable string format.
output_parser = StrOutputParser()


In [68]:
# Create the prompt template

# Creating a prompt template that instructs the AI to act as a movie information service agent.
# The prompt takes two parameters:
#   1. {context} - Relevant information retrieved from the document store.
#   2. {input} - The user's question.
# The model is instructed to base its answer solely on the provided context.

prompt = ChatPromptTemplate.from_template(
    """Respond to the following question based only on the provided context:

    <context>
    {context}
    </context>

    Question: {input}""",
    output_parser=output_parser  # The output parser ensures that the response is returned in a structured string format.
)




In [75]:
# Create the document processing chain
document_chain = create_stuff_documents_chain(llm, prompt) #prompt | llm


In [76]:
# Create a retriever from the vector store for fetching relevant documents
# See https://python.langchain.com/v0.1/docs/modules/data_connection/retrievers/vectorstore/
retriever = vector.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)



In [ ]:
# Invoke the retrieval chain to process the user's query
retrieval_chain.invoke({"input": "what are the top movies?"})


{'input': 'what are the top movies?',
 'context': [Document(id='7df32431-c553-4537-ac66-0df44975c1fd', metadata={}, page_content='Up is a Animation movie directed by Pete Docter. It stars Pete DocterBob PetersonTom McCarthy and was released in 2009.'),
  Document(id='6b74e22b-edd5-4c81-a590-5fdd6067aa02', metadata={}, page_content='From Up on Poppy Hill is a Animation movie directed by Gorô Miyazaki. It stars Tetsurô SayamaHayao MiyazakiKeiko Niwa and was released in 2011.'),
  Document(id='07bb74d8-f877-49b6-9c84-3e76fe04035c', metadata={}, page_content='Topsy-Turvy is a Biography movie directed by Mike Leigh. It stars Jim BroadbentAllan CordunerDexter Fletcher and was released in 1999.'),
  Document(id='45d07ce2-e537-4414-b468-4b99c1b023eb', metadata={}, page_content='Things to Come is a Drama movie directed by William Cameron Menzies. It stars H.G. Wells and was released in 1936.')],
 'answer': 'The provided context lists these movies: **Up (2009)**, **From Up on Poppy Hill (2011)**

In [79]:
# Perform adequate formatting to print the final response in a user readable format
retrieval_chain.invoke({"input": "what are the top movies of 2025?"})['answer']


'Based on the provided context, the top movie from **2025** is:\n\n- **Blade** — *Action* movie, directed by **Yann Demange**, released in **2025**.'

In [97]:
# Optional: Test the functionality using a Gradio UI (intermediate check)
import gradio as gr

def movie_chatbot(user_query):
    # Invoking the retrieval chain with the user's query to fetch relevant product information
    response = retrieval_chain.invoke({"input": user_query})['answer']
    prompt = f"Format the responses properly in {response}. Just return response in a human readable format. Do not include any other text except the response."
    # Sending the formatted prompt to the GPT-5.4-nano model for processing
    openai_response = client.chat.completions.create(
        model='gpt-5.4-nano',  # Using GPT-5.4-nano model for response generation
        messages=[{'role': 'user', 'content': prompt}]  # Providing the prompt to the model
    )
    # Extracting and returning the AI-generated response containing only the product names
    return openai_response.choices[0].message.content


chatbot_interface = gr.Interface(
    fn=movie_chatbot,
    inputs=gr.Textbox(label="Enter your query here"),
    outputs=gr.Textbox(label="Response"),
    title="Movie Chatbot",
    description="Ask me about movies!",
    theme="Glass",  # Optional: Choose a theme for the interface
    flagging_mode="never" 
)
chatbot_interface.launch()


/Users/uchegodfrey/Downloads/Ik_assignments/.venv/lib/python3.13/site-packages/gradio/interface.py:171: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  super().__init__(


* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


In [ ]:
# Define various agents - each performing a particular task using tool decorator


In [ ]:
# Define the orchestrator logic to run the agents appropriately


In [ ]:
# Check the edge cases and handle them appropriately


In [ ]:
# Create a UI using gradio or any other tool of your choice


# Test the performance of your bot with various test cases and refine your code!
